## Time-weighted Vector Retriever

 - 가장 최근에 이용된 문서를 기준으로 먼저 참고하도록 하여 답변의 최신화를 유지할 수 있음
 - 시간이 지난 만큼 페널티를 부여!

In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
!pip install langchain==0.1.16

In [ ]:
!pip install faiss-gpu-cu12

In [1]:
from langchain.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
ko_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs = encode_kwargs
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
from datetime import datetime, timedelta
import numpy
import faiss
from langchain.docstore import InMemoryDocstore
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.schema import Document
from langchain_community.vectorstores import FAISS

In [6]:
embedding_size = 768
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(ko_embedding, index, InMemoryDocstore({}), {})
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.99, k=1
)

In [7]:
yesterday = datetime.now() - timedelta(days=1)
retriever.add_documents(
    [Document(page_content="영어는 훌륭합니다.", metadata={"last_accessed_at": yesterday})]
)
retriever.add_documents([Document(page_content="한국어는 훌륭합니다")])

['0eeaf052-0a0d-4f0c-85c0-f4d3b9151baf']

In [8]:
retriever.get_relevant_documents("영어가 어려워요")

/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores.py:330: UserWarning: Relevance scores must be between 0 and 1, got [(Document(page_content='영어는 훌륭합니다.', metadata={'last_accessed_at': datetime.datetime(2026, 1, 18, 7, 29, 19, 25503), 'created_at': datetime.datetime(2026, 1, 19, 7, 29, 19, 25659), 'buffer_idx': 0}), np.float32(0.15314615)), (Document(page_content='한국어는 훌륭합니다', metadata={'last_accessed_at': datetime.datetime(2026, 1, 19, 7, 29, 19, 159520), 'created_at': datetime.datetime(2026, 1, 19, 7, 29, 19, 159520), 'buffer_idx': 1}), np.float32(-0.11495972))]
  warnings.warn(


[Document(page_content='한국어는 훌륭합니다', metadata={'last_accessed_at': datetime.datetime(2026, 1, 19, 7, 29, 19, 589897), 'created_at': datetime.datetime(2026, 1, 19, 7, 29, 19, 159520), 'buffer_idx': 1})]